[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/15-projects/p06-response-rating-tool/colab_quickstart.ipynb)

# P6 - Response-rating tool: quick start
Rate pairs of AI answers, export HH-RLHF-style preference data, and measure how well an automatic judge agrees with humans (Cohen's kappa). Runs on a normal CPU runtime - no GPU, no API key.

## 1. Get the code
In Colab this clones the course repo. If you are already inside the project folder (running locally), the clone is skipped.

In [1]:
import os, pathlib, subprocess, sys
PROJECT = "p06-response-rating-tool"
if pathlib.Path("src").is_dir() and pathlib.Path("README.md").exists():
    print("Project folder found - using the local copy, no clone needed.")
else:
    if not pathlib.Path("AI-Engineer").exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/vvvvvivekkk/AI-Engineer.git"], check=True)
    os.chdir(f"AI-Engineer/15-projects/{PROJECT}")
print("working in", os.getcwd())

Project folder found - using the local copy, no clone needed.
working in /home/claude/AI-Engineer/15-projects/p06-response-rating-tool


## 2. Install the pinned packages (Colab only)
Colab already has PyTorch with GPU support, so we skip the `torch` line to avoid replacing it. Locally you install with `pip install -r requirements.txt` in a venv.

In [2]:
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    reqs = [l for l in open("requirements.txt") if l.strip() and not l.startswith(("torch==", "#"))]
    open("/tmp/req.txt", "w").writelines(reqs)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/tmp/req.txt"], check=True)
print("ready")

ready


## 3. Make the data
100 prompts with two answers each (one has a planted flaw), plus simulated ratings from two pretend humans so the analysis works straight away.

In [3]:
!python -m src.make_data
!python -m src.simulate_raters

wrote 100 items -> data/candidates.json


wrote simulated ratings for 2 raters (+0 real ratings kept) -> data/ratings.jsonl


## 4. Run the rule-based judge and measure agreement
The judge is asked twice with the answers swapped to catch position bias.

In [4]:
!python -m src.judge
!python -m src.agreement

wrote 100 verdicts -> data/judge_rule.jsonl


pair                                        n  agree  kappa
sim_rater_1 vs sim_rater_2                100   0.75  0.563
judge_rule vs sim_rater_1                 100   0.86  0.737
judge_rule vs sim_rater_2                 100   0.82  0.672
sim_rater_1 vs gold                       100   0.86  0.737
sim_rater_2 vs gold                       100   0.82  0.672
judge_rule vs gold                        100    1.0    1.0
judge gave the same verdict in both orders on 100% of items
saved -> data/agreement_report_rule.json


## 5. Export chosen/rejected pairs
This is the file P10 can use for DPO training.

In [5]:
from src.ratings import DATA, export_preferences, load_candidates, read_jsonl
n = export_preferences(load_candidates(), read_jsonl(DATA / "ratings.jsonl"), DATA / "preferences_hh.jsonl", rater="sim_rater_1")
print(n, "pairs written")
print(open(DATA / "preferences_hh.jsonl").readline()[:300])

93 pairs written
{"chosen": "\n\nHuman: A shop sells pencils at 5 rupees each. Riya buys 6 pencils and pays with a 200 rupee note. How much change does she get?\n\nAssistant: Let's work it out step by step.\nCost of pencils = 6 x 5 = 30 rupees.\nChange = 200 - 30 = 170 rupees.\nFinal answer: 170", "rejected": "\n\nH


## 6. (Optional) the tiny local LLM as judge
Slow-ish and close to random with a 135M model - that is the lesson. Watch the position-consistency number.

In [6]:
!python -m src.judge --judge llm --limit 6
!python -m src.agreement --judge llm

Loading weights:   0%|                                 | 0/272 [00:00<?, ?it/s]

Loading weights:  24%|█████▌                 | 66/272 [00:00<00:00, 654.50it/s]

Loading weights:  53%|███████████▋          | 145/272 [00:00<00:00, 726.67it/s]

Loading weights:  80%|█████████████████▋    | 218/272 [00:00<00:00, 696.20it/s]

Loading weights: 100%|██████████████████████| 272/272 [00:00<00:00, 664.19it/s]


judged 1/6


... (2 outputs left out)

judged 4/6


judged 5/6


judged 6/6
wrote 6 verdicts -> data/judge_llm.jsonl


pair                                        n  agree  kappa
sim_rater_1 vs sim_rater_2                100   0.75  0.563
judge_llm vs sim_rater_1                    6    0.0    0.0
judge_llm vs sim_rater_2                    6  0.333    0.0
sim_rater_1 vs gold                       100   0.86  0.737
sim_rater_2 vs gold                       100   0.82  0.672
judge_llm vs gold                           6    0.0    0.0
judge gave the same verdict in both orders on 0% of items
saved -> data/agreement_report_llm.json


## 7. Run the tests

In [7]:
!python -m pytest -q tests

...

..

.

.                                                                  [100%]
7 passed in 2.57s


## 8. Open the rating app (Colab only)
Starts Streamlit in the background and opens it in a Colab window. Locally, run `streamlit run app.py` in a terminal instead.

In [8]:
if IN_COLAB:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"])
    import time; time.sleep(5)
    from google.colab import output
    output.serve_kernel_port_as_window(8501)
else:
    print("Not in Colab: run `streamlit run app.py` in a terminal.")

Not in Colab: run `streamlit run app.py` in a terminal.
